In [ ]:
import numpy as np
import torch
from pathlib import Path
import problem_config as cfg

device = "cpu"
dtype = cfg.dtype

# --- Paths relative to problem_config.py ---
base_dir   = Path(cfg.__file__).parent.resolve()
data_dir   = base_dir / "problem_data"
result_dir = base_dir / "result"
result_dir.mkdir(exist_ok=True)

# --- Load data ---
y_np = np.load(data_dir / "obs_path.npy")          # (T, m) or (T+1, m); we'll use y.shape[0] steps
y    = torch.tensor(y_np, dtype=dtype, device=device)

x_true_np = np.load(data_dir / "state_path.npy")   # used only for printing error per step

# --- Matrices ---
A = cfg.A.to(device=device, dtype=dtype)
H = cfg.H.to(device=device, dtype=dtype)
Q = cfg.Q.to(device=device, dtype=dtype)
R = cfg.R.to(device=device, dtype=dtype)

n = cfg.DIM_STATE
m = cfg.DIM_OBS
T = y.shape[0]

# --- KF prior ---
m_k = torch.zeros(n, dtype=dtype, device=device)   # m_0
P_k = torch.eye(n, dtype=dtype, device=device)     # P_0

# --- Storage ---
ms = torch.zeros(T-1, n, dtype=dtype, device=device)
Ps = torch.zeros(T-1, n, n, dtype=dtype, device=device)

I = torch.eye(n, dtype=dtype, device=device)

for k in range(T-1):
    ms[k] = m_k
    Ps[k] = P_k
    
    # ---- Forecast (time update) ----
    m_f = A @ m_k
    P_f = A @ P_k @ A.T + Q

    # ---- Analysis (measurement update) ----
    S = H @ P_f @ H.T + R                     # innovation covariance
    # Kalman gain without explicit inverse: K = P_f H^T S^{-1}
    # Solve S^T X^T = (H P_f^T)^T for X^T, then transpose -> equivalent to K
    K = torch.linalg.solve(S, (H @ P_f.T)).T

    innov = y[k+1] - (H @ m_f)
    m_a = m_f + K @ innov

    # Joseph stabilized covariance update:
    KH = K @ H
    P_a = (I - KH) @ P_f @ (I - KH).T + K @ R @ K.T


    # Prepare next step
    m_k, P_k = m_a, P_a

    # print error
    # Be lenient about x_true_np length (some datasets store T+1)
    x_true_k = x_true_np[k if k < len(x_true_np) else -1]
    print(f"Step {k+1}/{T} \t error = {np.linalg.norm(ms[k].cpu().numpy() - x_true_k):.4f}")

# Save results
out_path = result_dir / "kf.npz"
np.savez(out_path,
         posterior_mean=ms.cpu().numpy(),
         posterior_cov=Ps.cpu().numpy())
print(f"Saved KF posterior to: {out_path}")
